# Noite 2 · 1 de 2 · Ingestão de uma API

A primeira ingestão da noite, e a mais simples: uma API pública, sem chave. Na empresa, o dado não chega
pronto: alguém precisa ir buscar. Quando a fonte não tem conector, entra o Python, sempre em três passos:
**extrair** (pedir à API), **pousar** (guardar o arquivo no raw, do jeito que veio) e **carregar** (o
`read_files` transforma o arquivo em tabela).

**Pergunta de negócio:** quantos dias úteis a Vértice perde em novembro, bem quando começa o change freeze
da Black Friday (01/11)?

No Databricks: **New › Notebook**, nome `01_ingestao_api`, computação **serverless** (ela tem internet).
Cole uma célula de cada vez. Com lacunas para fazer junto: [`01_ingestao_api.py`](01_ingestao_api.py) · código pronto:
[`01_ingestao_api_gabarito.py`](01_ingestao_api_gabarito.py).

## Passo 1 · Extrair: pedir os feriados à API

In [0]:
import requests                # conversa com a API (HTTP)

url = "https://brasilapi.com.br/api/feriados/v1/2026"

# Objetivo: pedir os feriados à API e transformar a resposta em lista do Python.
# 1. resposta = requests.get(url, timeout=30)
# 2. resposta.raise_for_status()      -> se a API cair, para aqui
# 3. feriados = resposta.json()
# 4. imprima quantos feriados vieram (len) e mostre os 3 primeiros (feriados[:3])

# escreva seu código aqui
response = requests.get(url, timeout=30)
response.raise_for_status()
feriados = response.json()
print(f"Quantidade de feriados em 2026: {len(feriados)}")

## Passo 2 · Pousar: o JSON numa pasta nova do raw

Cada execução grava um arquivo **novo**, com a data e a hora no nome. O raw guarda o histórico de
tudo o que chegou: dá para saber o que a API respondia em cada momento.

In [0]:
import json, os
from datetime import datetime

pasta = "/Volumes/lakehouse_vertice/bronze/raw/feriados"
os.makedirs(pasta, exist_ok=True)                        # a pasta nova precisa existir

agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")     # ex.: 2026-10-20_19-40-00 (horário UTC)

# Objetivo: gravar a lista `feriados` no arquivo f"{pasta}/feriados_{agora}.json".
# Dica: with open(caminho, "w") as arquivo:
#           json.dump(feriados, arquivo, ensure_ascii=False)
# Depois, liste a pasta com sorted(os.listdir(pasta)).
    
# escreva seu código aqui
caminho = f"{pasta}/feriados_{agora}.json"

with open(caminho, 'w') as f:
    json.dump(feriados, f, ensure_ascii=False)
    
print(sorted(os.listdir(pasta)))

## Passo 3 · Carregar: o raw vira a bronze

O mesmo `read_files` da noite 1, lendo **todos** os arquivos da pasta. Duas colunas a mais contam de onde
e de quando veio cada linha.

In [0]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
COMMENT 'Feriados nacionais de 2026, da BrasilAPI. Uma captura por execução do job.'
AS SELECT date AS data, name AS feriado, type AS tipo, weekday AS dia_da_semana,
          _metadata.file_name              AS arquivo,
          _metadata.file_modification_time AS coletado_em
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/feriados/', format => 'json', multiLine => true)

## Passo 4 · Conferir

In [0]:
%sql
-- uma linha por captura: 14 feriados em cada uma
SELECT arquivo, coletado_em, COUNT(*) AS feriados
FROM lakehouse_vertice.bronze.feriados
GROUP BY arquivo, coletado_em
ORDER BY coletado_em DESC

In [0]:
%sql
-- a resposta para o negócio, na captura mais recente:
-- 02/11 (segunda), 15/11 (domingo) e 20/11 (sexta): 2 dias úteis a menos em novembro
SELECT data, feriado, dia_da_semana
FROM lakehouse_vertice.bronze.feriados
WHERE arquivo = (SELECT MAX(arquivo) FROM lakehouse_vertice.bronze.feriados)
  AND Date(data) BETWEEN current_date AND current_date + make_dt_interval(30)
ORDER BY data